# Day 1 · Module 3b (companion) — Drive the CXR Agent with a REAL LLM (vLLM)

**What this is.** The same minimal CXR agent you built in Module 3b, but the controller is now a **real language model** served from your instructor's **vLLM** endpoint instead of the offline `MockLLM`. The tools, the ReAct loop, and the transcript are byte-for-byte the same — *only* `llm.decide` changes.

**Why it matters.** The `MockLLM` is deterministic: it always calls the right tools in the right order. A real model does **not**. It may stop early, skip a required tool, reorder calls, or write a fluent answer that cites a tool it never called. Reading the transcript — the skill from Module 3b — is what catches that.

> **Student notebook** · kernel **AImed**. Needs the endpoint URL your instructor shares. No endpoint? Every cell falls back to the offline `MockLLM` so the notebook still runs.

### References
- **ReAct loop** (Yao et al. 2023): reason -> act (call a tool) -> observe -> repeat. Same loop as Module 3b.
- **vLLM** (Kwon et al. 2023): a high-throughput OpenAI-compatible server. One GPU server backs the whole cohort.
- *Today's goal:* does a real model reliably follow the protocol your `MockLLM` enforced by construction — and how would you know?

In [ ]:
# --- Environment setup (run me first; you don't need to edit this) ---
# Finds the shared course 'scripts/' folder and puts it on Python's import path so the
# later cells can do `import minimal_cxr_agent` and `from common import llm_backends`.
import sys, pathlib, os
cands = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]   # current dir + all parent dirs
if os.environ.get('PROJECT_DIR'):                            # + the shared course dir if it is set
    cands.append(pathlib.Path(os.environ['PROJECT_DIR']))
scripts = next((b/'scripts' for b in cands                   # first candidate that has scripts/common/
                if (b/'scripts'/'common'/'aimed_config.py').exists()), None)
# if this assert fires, your kernel isn't seeing the course tree -> relaunch the AImed kernel
assert scripts, 'run from inside the course tree, or set PROJECT_DIR to the shared course dir'
sys.path.insert(0, str(scripts))                             # make those scripts importable below
from common import aimed_config as cfg                       # shared paths + cache configuration
cfg.setup_caches()                                           # point HF/torch caches at shared storage
print('environment OK  ->', cfg.PROJECT_DIR)

## 1 · Connect to the shared LLM endpoint

Your instructor starts one vLLM server (`sbatch slurm/serve_vllm_agent.sbatch`) and gives you a URL like `http://gpu0123:8000/v1`. The server also writes that URL to `logs/vllm_endpoint.env` in the shared course tree — so if you're on the cluster, the next cell usually finds it **with no edits**. If it doesn't, paste the URL where marked.

In [ ]:
# Point the agent at your instructor's vLLM server.
import os, pathlib

# (a) If the instructor's server wrote a shared endpoint file, load it automatically.
env_file = pathlib.Path(os.environ.get('PROJECT_DIR', '.')) / 'logs' / 'vllm_endpoint.env'
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if '=' in line and not line.lstrip().startswith('#'):
            k, v = line.split('=', 1); os.environ[k.strip()] = v.strip()
    print('loaded endpoint from', env_file)

# (b) ...otherwise EDIT the line below with the URL your instructor gives you.
os.environ.setdefault('OPENAI_BASE_URL', 'http://CHANGE-ME:8000/v1')   # <-- paste it here if no shared file
os.environ.setdefault('OPENAI_API_KEY', 'EMPTY')                       # vLLM ignores the key, but the client wants one
os.environ.setdefault('AIMED_LLM_MODEL', 'Qwen/Qwen3-4B')              # the name the server advertises

print('OPENAI_BASE_URL =', os.environ['OPENAI_BASE_URL'])
print('AIMED_LLM_MODEL =', os.environ['AIMED_LLM_MODEL'])

In [ ]:
# Health check: is the endpoint actually reachable, and what is it serving?
import urllib.request, json
ENDPOINT_OK = False
base = os.environ.get('OPENAI_BASE_URL', '').rstrip('/')
try:
    with urllib.request.urlopen(base + '/models', timeout=5) as r:
        served = [m['id'] for m in json.load(r).get('data', [])]
    ENDPOINT_OK = True
    print('endpoint reachable ->', base)
    print('served model(s)    ->', served)
except Exception as e:
    print(f'endpoint NOT reachable: {e}')
    print('-> falling back to the offline MockLLM so this notebook still runs end-to-end.')
    print('   get the URL from your instructor, re-run the connect cell, then re-run this one.')

## 2 · Run the agent with the real model

`llm_backends.get_llm('openai')` builds an OpenAI-compatible client from the env vars you just set; `get_llm('mock')` is the offline rule-based stand-in. We pick `openai` only if the health check passed. **Everything below this line is identical to Module 3b** — that is the whole point: the loop never knew or cared which controller it was talking to.

In [ ]:
import minimal_cxr_agent as AG
from common import llm_backends
import json

llm = llm_backends.get_llm('openai' if ENDPOINT_OK else 'mock')
tools, img, feats = AG.build_stub_tools()      # same cheap deterministic tools as Module 3b
print('LLM backend     :', llm.name, '' if ENDPOINT_OK else '(offline fallback)')
print('the question    :', AG.QUESTION)

answer, transcript = AG.run_agent(AG.QUESTION, img, feats, tools, llm)

print('\nTRANSCRIPT (the auditable trace) ---------------------------------')
for i, step in enumerate(transcript):
    print(f'  step {i}: {json.dumps(step)}')
print('\nFINAL ANSWER:', answer)
print('\ntools actually called, in order:', AG.tools_called(transcript))
print('Gate-3b:', AG.gate3b_check(transcript))

## 3 · Same question, two controllers — Mock vs real

Run the identical question through the deterministic `MockLLM` and the real model and line up their trajectories. The Mock calls `classify_cxr -> gradcam_region -> tabular_risk` every time. Watch whether the real model does — calling all required tools, in a sensible order, before it answers.

In [ ]:
from common import llm_backends
import minimal_cxr_agent as AG

backends = [('MockLLM (offline, deterministic)', llm_backends.get_llm('mock'))]
if ENDPOINT_OK:
    backends.append((f"real model ({os.environ['AIMED_LLM_MODEL']})", llm_backends.get_llm('openai')))

for label, lm in backends:
    ans, tr = AG.run_agent(AG.QUESTION, img, feats, tools, lm)
    g = AG.gate3b_check(tr)
    print(f'--- {label} ---')
    print('  tools called :', AG.tools_called(tr))
    print('  gate passed  :', g['passed'], '| required tools present:', g['have_required'], '| order ok:', g['order_ok'])
    print('  answer       :', ans[:200])
    print()
if not ENDPOINT_OK:
    print('(no endpoint -> only the Mock ran. Connect the endpoint to see the contrast.)')

## 4 · Real models are non-deterministic — run it a few times

`MockLLM` gives the same trajectory on every run. A real model, even at `temperature=0`, can vary run to run (and certainly across questions). Run the agent several times and tabulate the tool sequence and whether Gate-3b passed. **Variation in the execution path is exactly what makes an agent hard to validate** — you are not testing one output, you are testing a different program trace each time.

In [ ]:
import minimal_cxr_agent as AG
from common import llm_backends

N = 5
lm = llm_backends.get_llm('openai' if ENDPOINT_OK else 'mock')
print(f'running the agent {N}x with backend = {lm.name}'
      + ('' if ENDPOINT_OK else '  (Mock is deterministic, so every run is identical -- that is the contrast)'))
print()
rows = []
for k in range(N):
    ans, tr = AG.run_agent(AG.QUESTION, img, feats, tools, lm)
    g = AG.gate3b_check(tr)
    seq = ' -> '.join(AG.tools_called(tr)) or '(none)'
    rows.append((k, g['passed'], seq))
    print(f'  run {k}: gate={"PASS" if g["passed"] else "RETRY"}  | path: {seq}')
distinct = len({s for _, _, s in rows})
print(f'\n{distinct} distinct execution path(s) across {N} runs; '
      f'{sum(p for _, p, _ in rows)}/{N} passed Gate-3b.')

## 5 · Your turn (no auto-grader) — audit the REAL trajectory

In Module 3b you audited the Mock's clean transcript. A real model's free-text answer is harder: it can *claim* a localization without ever calling `gradcam_region`, or assert pneumonia while `classify_cxr` returned a low probability. Run the quick audit below on a real trajectory and judge it: are the flags right? What does it miss that a human reading the prose would catch?

In [ ]:
import minimal_cxr_agent as AG

def quick_audit(answer, transcript):
    """Heuristic checks on a (possibly real-LLM) trajectory. Returns a sorted flag list."""
    called = AG.tools_called(transcript)
    a = (answer or '').lower()
    prob = next((s['result'].get('pneumonia_prob') for s in transcript
                 if isinstance(s.get('result'), dict) and 'pneumonia_prob' in s['result']), None)
    flags = []
    for t in ('classify_cxr', 'tabular_risk'):                       # 3-part question needs both
        if t not in called: flags.append(f'missing_required_tool:{t}')
    if any(w in a for w in ('lobe', 'zone', 'region')) and 'gradcam_region' not in called:
        flags.append('claims_localization_without_calling_gradcam')   # a hallucinated finding
    if prob is not None and prob < 0.5 and 'pneumonia' in a and 'no pneumonia' not in a:
        flags.append('asserts_pneumonia_despite_low_prob')            # an ignored conflict
    return sorted(flags)

answer, transcript = AG.run_agent(AG.QUESTION, img, feats, tools,
                                  llm_backends.get_llm('openai' if ENDPOINT_OK else 'mock'))
print('answer  :', answer[:240])
print('flags   :', quick_audit(answer, transcript) or '(none — but read the prose yourself before trusting that)')

### Discussion
- The Mock passed Gate-3b by construction; the real model sometimes did not. If your *acceptance test* only ever ran the Mock, what would you have failed to learn about the system you actually ship?
- A real model wrote a confident, well-formed answer. Which parts of it are grounded in a tool result, and which are the model's prose? How do you check that **automatically**, across thousands of cases?
- One shared GPU server backs the whole class. What changes when the controller is a hosted model you don't control — latency, cost, reproducibility, privacy of the data you send it?

### Notes & stretch
1. **Turn up the temperature.** In `common/llm_backends.py`, `OpenAICompatLLM` defaults to `temperature=0`. Build one with `OpenAICompatLLM(temperature=1.0)` and re-run section 4 — the execution paths should spread out. More creativity, less protocol-following.
2. **A different question.** `AG.run_agent('Where is the abnormality?', img, feats, tools, lm)` — does the model still call `classify_cxr` first, or jump straight to localizing?
3. **Any OpenAI-compatible endpoint works.** Point `OPENAI_BASE_URL` at Ollama or the real OpenAI API (with a real key) instead of vLLM — the agent code does not change.
4. **Real tools.** Combine with `--real-tools` (see `minimal_cxr_agent.py`) to drive your actually-trained Module-1/2 models with the real LLM. The transcript reads the same; that is the point.

**Instructor:** start the server with `sbatch slurm/serve_vllm_agent.sbatch`; it self-tests the agent, prints the endpoint, and writes `logs/vllm_endpoint.env` for students to auto-load.